<a href="https://colab.research.google.com/github/AhnafTouseef/Blender-Colab-render-pipline/blob/main/Rendering_pipeline_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **bold text**

In [26]:
from pathlib import Path
import shutil


# -----------------------------
# Create
# -----------------------------

def new_folder(path):
    """
    Create a new folder.
    """
    Path(path).mkdir(parents=True, exist_ok=True)
    return f"New folder: {path}"


def new_file(path, content=""):
    """
    Create a new file.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)
    return f"New file: {path}"



# -----------------------------
# File operations
# -----------------------------

def copy(source, destination):
    """
    Copy file or folder.
    """

    source = Path(source)

    if source.is_dir():
        shutil.copytree(
            source,
            destination,
            dirs_exist_ok=True
        )
    else:
        shutil.copy2(
            source,
            destination
        )

    return f"Copied: '{source}' to: '{destination}'"



def move(source, destination):
    """
    Move file or folder.
    """
    shutil.move(str(source),str(destination))
    return f"Moved: '{source}' to: '{destination}'"


def move_from(source, destination):
    try:
        delete(destination+'/.ipynb_checkpoints')
    except:
      pass
    items = list(Path(source).iterdir())
    for item in items:
        move(item, destination)
    return f"Cut to: '{destination}'"


def copy_from(source, destination):
    try:
        delete(destination+'/.ipynb_checkpoints')
    except:
      pass
    items = list(Path(source).iterdir())
    for item in items:
        copy(item, destination)
    return f"Copied to: '{destination}'"


def rename(path, new_name):
    """
    Rename a file or folder.
    """

    path = Path(path)

    new_path = path.parent / new_name

    return f" Renamed: '{path}' to: '{path.rename(new_path)}'"



def delete(path):
    """
    Delete file or folder.
    """

    path = Path(path)

    if path.is_dir():
        shutil.rmtree(path)

    else:
        path.unlink()
    return f"Deleted: '{path}'"



def delet_content(path):
    if path.is_dir():
        for item in Path("/content/drive/MyDrive/Render_Farm/output").iterdir():
            delete(item)
        return f"Deleted contents of: '{path}'"
    else:
        return f"Not a directory: '{path}'"




# -----------------------------
# Navigation
# -----------------------------

def exists(path):
    return Path(path).exists()



def open_folder(path):
    """
    Return contents of folder.
    """
    import pandas as pd
    File = []
    Address = []
    df = pd.DataFrame(columns=["Address", "Files"])
    try:
        for item in Path(path).iterdir():
            Address.append(str(item))
            File.append(item.name)
        df = pd.DataFrame({"Address": Address, "Files": File})
    except Exception as e:
        print(str(e))
    return df.style.set_properties(subset=['Address', "Files"], **{'text-align': 'left'})\
             .set_table_styles([dict(selector='th', props=[('text-align', 'center')])])



def search(location, keyword):
    """
    Search files and folders.
    """

    results = []

    for item in Path(location).rglob("*"):

        if keyword.lower() in item.name.lower():
            results.append(item)

    return results



def find(location, extension):
    """
    Find files by extension.
    """

    return list(
        Path(location).rglob(f"*.{extension}")
    )



# -----------------------------
# Information
# -----------------------------

def properties(path):

    p = Path(path)

    # Helper function to format size
    def format_size(size_bytes):
        if size_bytes == 0:
            return "0 B"
        size_name = ("B", "KB", "MB", "GB", "TB")
        i = int(math.floor(math.log(size_bytes, 1024)))
        p = math.pow(1024, i)
        s = round(size_bytes / p, 2)
        return f"{s} {size_name[i]}"

    import math

    size_in_bytes = p.stat().st_size

    return {
        "name": p.name,
        "location": str(p.parent),
        "type": "folder" if p.is_dir() else "file",
        "size": format_size(size_in_bytes)
    }

try:
  delete('/content/sample_data')
except:
  pass

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# New section

In [3]:
!wget https://download.blender.org/release/Blender4.5/blender-4.5.1-linux-x64.tar.xz
!tar xf blender-4.5.1-linux-x64.tar.xz

--2026-09-16 05:17:48--  https://download.blender.org/release/Blender4.5/blender-4.5.1-linux-x64.tar.xz
Resolving download.blender.org (download.blender.org)... 172.66.172.236, 104.20.41.146, 2606:4700:10::6814:2992, ...
Connecting to download.blender.org (download.blender.org)|172.66.172.236|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 375503560 (358M) [application/octet-stream]
Saving to: ‘blender-4.5.1-linux-x64.tar.xz’

blender-4.5.1-linux 100%[===================>] 358.11M   184MB/s    in 1.9s    

2026-09-16 05:17:50 (184 MB/s) - ‘blender-4.5.1-linux-x64.tar.xz’ saved [375503560/375503560]



In [4]:
delete('/content/blender-4.5.1-linux-x64.tar.xz')

"Deleted: '/content/blender-4.5.1-linux-x64.tar.xz'"

# Configs

In [27]:
import os

# Configuration parameters for Blender rendering
TOTAL_WORKER = 5 # @param {type:"integer"}
RENDER_FILE = "Cactus.blend" # @param {type:"string"}

# Output settings
RENDER_ENGINE = 'CYCLES' # @param ["CYCLES", "BLENDER_EEVEE", "BLENDER_WORKBENCH"] {type:"string"}
ADAPTIVE_SAMPLING = False # @param {type:"boolean"}
ADAPTIVE_THRESHOLD = 0.01 # @param {type:"number"}
RENDER_SAMPLES = 1 # @param {type:"integer"}
RENDER_ANIMATION = False # @param {type:"boolean"}
FPS = 24 # @param {type:"integer"}
RESOLUTION_PERCENT = 20 # @param {type:"integer"}
USE_DENOISE = False # @param {type:"boolean"}
DENOISER = "OPTIX" # @param ["OPTIX", "OPENIMAGEDENOISE", "NONE"] {type:"string"}
USE_PERSISTENT_DATA = True # @param {type:"boolean"}
OUTPUT_FORMAT = "PNG" # @param ["PNG", "JPEG", "OPEN_EXR", "FFMPEG"] {type:"string"}
SPATIAL_SPLITS = True # @param {type:"boolean"}

# Directory for configuration file
CONFIG_DIR = "/content/drive/MyDrive/Render_Farm"

# Ensure the directory exists
os.makedirs(CONFIG_DIR, exist_ok=True)

# Construct the Blender Python API content for render settings
render_settings_content = f"""
import bpy;
S = bpy.context.scene;
S.render.engine = '{RENDER_ENGINE}';
S.cycles.use_adaptive_sampling = {ADAPTIVE_SAMPLING};
S.cycles.adaptive_threshold = {ADAPTIVE_THRESHOLD};
S.cycles.samples = {RENDER_SAMPLES};
S.render.fps = {FPS};
S.render.resolution_percentage = {RESOLUTION_PERCENT};
S.cycles.use_denoising = {USE_DENOISE};
S.cycles.denoiser = '{DENOISER}';
S.render.use_persistent_data = {USE_PERSISTENT_DATA};
S.render.image_settings.file_format = '{OUTPUT_FORMAT}';
S.cycles.debug_use_spatial_splits = {SPATIAL_SPLITS};
bpy.ops.render.render(animation={RENDER_ANIMATION}, use_viewport=True);
"""

# Save the Blender API settings to a Python file
render_settings_file_path = os.path.join(CONFIG_DIR, 'render_settings.py')
with open(render_settings_file_path, 'w') as f:
    f.write(render_settings_content)

# Configure JSON file
with open(r'/content/drive/MyDrive/Render_Farm/config.json', 'w') as f:
  string = '{' + f'\n"total_workers" : {TOTAL_WORKER}, \n"blend_file" : "{RENDER_FILE}"\n' + '}'
  f.write(string)

print(f"Blender render settings saved to {render_settings_file_path}")

Blender render settings saved to /content/drive/MyDrive/Render_Farm/render_settings.py


# Main Code

In [28]:
import os
import sys
import json
import re
import subprocess
from IPython.display import clear_output

# ============================================================
# SETTINGS
# ============================================================

MAIN_DIRECTORY = "/content/drive/MyDrive/Render_Farm"

CONFIG_FILE = f"{MAIN_DIRECTORY}/config.json"
REGISTER_DIR = f"{MAIN_DIRECTORY}/register"
OUTPUT_DIR = f"{MAIN_DIRECTORY}/output"

BLENDER_EXEC = "/content/blender-4.5.1-linux-x64/blender"

# Output settings
OUTPUT_FORMAT = "PNG"          # PNG / JPEG / OPEN_EXR
RESOLUTION_PERCENT = 20     # In percentage %
RENDER_SAMPLES = 1         # None = use blend settings


os.makedirs(REGISTER_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================

with open(CONFIG_FILE, 'r') as f:
    config = json.load(f)

TOTAL_WORKERS = int(config["total_workers"])
BLEND_PATH = f"{MAIN_DIRECTORY}/{config['blend_file']}"


# ============================================================
# WORKER
# ============================================================

def register_worker():
    ids = [int(os.path.splitext(x)[0]) for x in os.listdir(REGISTER_DIR) if os.path.splitext(x)[0].isdigit()]
    worker = max(ids) + 1 if ids else 1

    if worker == TOTAL_WORKERS:
        print(f"{worker}th worker arived. Cleaning registry")
        for file in os.listdir(REGISTER_DIR):
            os.remove(f"{REGISTER_DIR}/{file}")
        return worker
    else:
        open(f"{REGISTER_DIR}/{worker}.txt", "w").close()
        print(f"Worker {worker} registered")
        return worker


worker_id = register_worker()
# worker_id = 4


# ============================================================
# BLENDER DATA
# ============================================================

def get_scene_data():
    code = "import bpy; c=bpy.context.scene; print(f'START:{c.frame_start}\\nEND:{c.frame_end}')"

    res = subprocess.run(
        f'/content/blender-4.5.1-linux-x64/blender -b {BLEND_PATH} --python-expr "{code}"',
        shell=True,
        text=True,
        capture_output=True,
    )

    # Extract numbers directly using split
    print(res.stdout)
    start = res.stdout.split("START:")[1].split("\n")[0]
    end = res.stdout.split("END:")[1].split("\n")[0]
    return {"start_frame":start, "end_frame":end}


scene = get_scene_data()

START_FRAME = int(scene["start_frame"])
END_FRAME = int(scene["end_frame"])



def split_frames(worker_index, total_workers, start_frame, end_frame):
    total_frame = end_frame - start_frame + 1
    division, remainder = divmod(total_frame, total_workers)
    size = division + (1 if worker_index < remainder else 0)
    offset = worker_index * division + min(worker_index, remainder)
    return start_frame + offset, start_frame + offset + size - 1


my_start, my_end = split_frames(worker_id - 1, TOTAL_WORKERS, START_FRAME, END_FRAME)



# ============================================================
# BLENDER COMMAND
# ============================================================

python_override = f'''
import bpy;
S = bpy.context.scene;
S.cycles.samples = {RENDER_SAMPLES};
S.render.resolution_percentage = {RESOLUTION_PERCENT};
S.render.image_settings.file_format = {OUTPUT_FORMAT};
S.render.use_persistent_data = True;
S.cycles.use_denoising = True;
S.cycles.denoiser = "OPTIX";
bpy.context.object.data.dof.use_dof = False;

'''



# cmd = [BLENDER_EXEC, "-b", BLEND_PATH, "--python-expr", python_override, "-o", f"{OUTPUT_DIR}/frame_", "-s", str(my_start), "-e", str(my_end), "-a"]
# cmd = [BLENDER_EXEC, "-b", BLEND_PATH, "--python-expr", python_override, "-o", f"{OUTPUT_DIR}/frame_####", "-s", str(my_start), "-e", str(my_end), "-a", "--", "--cycles-device", "OPTIX"]
finder = re.compile(r"Fra:(\d+)")
cmd = [BLENDER_EXEC, "-b", BLEND_PATH,"--python-expr", python_override,"-o", f"{OUTPUT_DIR}/Frame_","-F","PNG","-s", str(my_start), "-e", str(my_end),"-a", "--", "--cycles-device", "OPTIX"]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
# ============================================================
# LIVE MONITOR
# ============================================================

current_frame = my_start

for line in process.stdout:
    # clear_output(wait=True)
    # print(line)

    try:
      frame = finder.search(line).group(1)
    except:
      continue
    # print(frame)
    if frame:
        current_frame = int(frame)

    frame_ratio = (current_frame - my_start + 1) / (my_end - my_start + 1)

    frame_bar = "█" * int(frame_ratio * 30) + "-" * (30 - int(frame_ratio * 30))

    clear_output(wait=True)
    print(f"Project name: {config['blend_file']}, Start Frame: {START_FRAME}, End Frame{END_FRAME}")

    print("=" * 50)
    print(f"Worker {worker_id}/{TOTAL_WORKERS}")
    print(f"Frames {my_start} -> {my_end}")
    print(f"Output {OUTPUT_DIR}")
    print("=" * 50)

    print("=" * 50)
    print("Blender Render Monitor")
    print("=" * 50)
    print(f"Frame   : {current_frame}/{my_end}")
    print(f"Overall : [{frame_bar}] {frame_ratio * 100:.2f}%")




process.wait()


if process.returncode == 0:
    print("\n✅ Render finished")
else:
    print("\n❌ Render failed")

Project name: Cactus.blend, Start Frame: 31, End Frame129
Worker 5/5
Frames 111 -> 129
Output /content/drive/MyDrive/Render_Farm/output
Blender Render Monitor
Frame   : 129/129
Overall : [██████████████████████████████] 100.00%

✅ Render finished


'F'